# 03. Jun Park - Word2Vec + Siamese LSTM Data-Only Experiment

This notebook is the final data-only version for Jun Park's part.
It trains the Word2Vec + Siamese LSTM model and saves reusable CSV/JSON/NPY outputs for teammates.

**Important:** This notebook intentionally does **not** generate plots. Final figures can be created later from the saved tables, especially `training_history.csv`, `threshold_comparison_val_test.csv`, `pair_similarity_scores.csv`, and `kmeans_lstm_pca_points.csv`.


In [ ]:
# 03. Jun Park - Word2Vec + Siamese LSTM Data-Only Experiment
# -------------------------------------------------------------
# Purpose
#   - Rebuild the LSTM-ready data from Kaggle train.csv if needed.
#   - Train Jun Park's Word2Vec + Siamese LSTM duplicate-question detector.
#   - Save clean, reusable CSV/JSON/NPY outputs for final benchmarking and plotting.
#   - This script intentionally DOES NOT generate plots.
#
# Run from the repository root:
#   python 03_siamese_lstm_jun_data_only.py
#
# Required input:
#   data/train.csv  or  data/train.csv.zip
#
# Main outputs:
#   outputs/results/siamese_lstm_results.csv
#   outputs/jun/tables/*.csv
#   outputs/jun/arrays/*.npy
#   outputs/jun/models/*.keras
#   outputs/jun/metadata/*.json

from __future__ import annotations

from pathlib import Path
from collections import Counter
import json
import os
import pickle
import platform
import random
import re
import subprocess
import sys
import time
import warnings
import zipfile

warnings.filterwarnings("ignore")

In [ ]:
# =========================
# 0. Repository / directory setup

In [ ]:
# =========================

REPO_URL = "https://github.com/hykiop63/CSE36301_ML_final.git"
REPO_NAME = "CSE36301_ML_final"
ORIGINAL_CWD = Path.cwd()


def in_colab() -> bool:
    try:
        import google.colab  # type: ignore  # noqa: F401
        return True
    except Exception:
        return Path("/content").exists()


# If opened in Colab outside the repo, clone the team repository.
if not Path("01_preprocessing.ipynb").exists() and in_colab():
    repo_dir = Path("/content") / REPO_NAME
    if not repo_dir.exists():
        print(f"Cloning repository: {REPO_URL}")
        subprocess.run(["git", "clone", REPO_URL, str(repo_dir)], check=True)
    os.chdir(repo_dir)
elif Path(REPO_NAME).exists() and not Path("01_preprocessing.ipynb").exists():
    os.chdir(Path(REPO_NAME))

ROOT = Path.cwd()
print("Repository root:", ROOT)

# Output layout. Plot directory is intentionally not created.
DATA_DIR = ROOT / "data"
SPLIT_DIR = ROOT / "splits"
LSTM_DIR = ROOT / "outputs" / "for_lstm"
BERT_DIR = ROOT / "outputs" / "for_bert"
JUN_DIR = ROOT / "outputs" / "jun"
TABLE_DIR = JUN_DIR / "tables"
ARRAY_DIR = JUN_DIR / "arrays"
MODEL_DIR = JUN_DIR / "models"
META_DIR = JUN_DIR / "metadata"
RESULT_DIR = ROOT / "outputs" / "results"

for directory in [DATA_DIR, SPLIT_DIR, LSTM_DIR, BERT_DIR, JUN_DIR, TABLE_DIR, ARRAY_DIR, MODEL_DIR, META_DIR, RESULT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

In [ ]:
# =========================
# 1. Config

In [ ]:
# =========================

RANDOM_STATE = 42

# Data / preprocessing
MAX_SEQ_LEN = 50
MIN_WORD_COUNT = 2
REBUILD_PREPROCESS = True  # Final clean run. After first run, set False if you want to reuse outputs/for_lstm.

# Word2Vec
USE_WORD2VEC = True
EMBED_DIM = 100
W2V_WINDOW = 5
W2V_MIN_COUNT = 1
W2V_EPOCHS = 5
W2V_PAIR_LIMIT = None  # None = use all training pairs. For a quick test: 100000.

# Siamese LSTM
LSTM_UNITS = 64
DENSE_UNITS = 128
DROPOUT_RATE = 0.30
LEARNING_RATE = 1e-3
BATCH_SIZE = 512
EPOCHS = 8
PATIENCE = 2
TRAIN_LIMIT = None  # None = use full train split. For a quick test: 50000.

# Data export for teammates
EXPORT_TEXT_PREDICTIONS = True
EXPORT_FULL_TEST_EMBEDDINGS = True
KMEANS_SAMPLE_N = 5000
KMEANS_K = 5
PCA_COMPONENTS = 2

CONFIG = {
    "random_state": RANDOM_STATE,
    "max_seq_len": MAX_SEQ_LEN,
    "min_word_count": MIN_WORD_COUNT,
    "rebuild_preprocess": REBUILD_PREPROCESS,
    "use_word2vec": USE_WORD2VEC,
    "embedding_dim": EMBED_DIM,
    "word2vec_window": W2V_WINDOW,
    "word2vec_min_count": W2V_MIN_COUNT,
    "word2vec_epochs": W2V_EPOCHS,
    "word2vec_pair_limit": W2V_PAIR_LIMIT,
    "lstm_units": LSTM_UNITS,
    "dense_units": DENSE_UNITS,
    "dropout_rate": DROPOUT_RATE,
    "learning_rate": LEARNING_RATE,
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "patience": PATIENCE,
    "train_limit": TRAIN_LIMIT,
    "export_text_predictions": EXPORT_TEXT_PREDICTIONS,
    "export_full_test_embeddings": EXPORT_FULL_TEST_EMBEDDINGS,
    "kmeans_sample_n": KMEANS_SAMPLE_N,
    "kmeans_k": KMEANS_K,
    "pca_components": PCA_COMPONENTS,
}

random.seed(RANDOM_STATE)

In [ ]:
# =========================
# 2. Locate Kaggle train.csv

In [ ]:
# =========================


def extract_zip_to_data(zip_path: Path, target_dir: Path) -> None:
    """Extract a zip file into data/. Handles Kaggle's nested train.csv.zip shape."""
    print(f"Extracting {zip_path} -> {target_dir}")
    target_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(target_dir)

    nested_train_zip = target_dir / "train.csv.zip"
    if nested_train_zip.exists() and not (target_dir / "train.csv").exists():
        print(f"Extracting nested {nested_train_zip}")
        with zipfile.ZipFile(nested_train_zip, "r") as zf:
            zf.extractall(target_dir)


def find_or_upload_train_csv() -> Path:
    candidate_dirs = [
        ROOT / "data",
        ROOT,
        ORIGINAL_CWD / "data",
        ORIGINAL_CWD,
        Path("/content/data"),
        Path("/content"),
        Path("/kaggle/input/quora-question-pairs"),
    ]

    for directory in candidate_dirs:
        candidate = directory / "train.csv"
        if candidate.exists():
            return candidate.resolve()

    zip_names = ["train.csv.zip", "quora-question-pairs.zip", "quora-question-pairs-data.zip"]
    for directory in candidate_dirs:
        for zip_name in zip_names:
            candidate_zip = directory / zip_name
            if candidate_zip.exists():
                extract_zip_to_data(candidate_zip, DATA_DIR)
                train_path = DATA_DIR / "train.csv"
                if train_path.exists():
                    return train_path.resolve()

    for directory in candidate_dirs:
        if directory.exists():
            for zip_path in directory.glob("*.zip"):
                extract_zip_to_data(zip_path, DATA_DIR)
                train_path = DATA_DIR / "train.csv"
                if train_path.exists():
                    return train_path.resolve()

    if in_colab():
        print("Could not find train.csv. Upload Kaggle train.csv or train.csv.zip now.")
        from google.colab import files  # type: ignore

        uploaded = files.upload()
        for filename in uploaded:
            src = Path(filename)
            if src.suffix == ".zip":
                extract_zip_to_data(src, DATA_DIR)
            elif src.name == "train.csv":
                os.replace(str(src), str(DATA_DIR / "train.csv"))

        train_path = DATA_DIR / "train.csv"
        if train_path.exists():
            return train_path.resolve()

    raise FileNotFoundError("Put Kaggle Quora train.csv or train.csv.zip under data/.")


DATA_PATH = find_or_upload_train_csv()
print("Using data file:", DATA_PATH)

In [ ]:
# =========================
# 3. Integrated preprocessing for Jun's LSTM input

In [ ]:
# =========================

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

REQUIRED_LSTM_FILES = [
    "train_q1.npy", "train_q2.npy", "train_y.npy",
    "val_q1.npy", "val_q2.npy", "val_y.npy",
    "test_q1.npy", "test_q2.npy", "test_y.npy",
    "word2idx.pkl", "idx2word.pkl",
]

PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"
PAD_ID = 0
UNK_ID = 1

CONTRACTIONS = {
    "what's": "what is", "what're": "what are", "who's": "who is",
    "where's": "where is", "when's": "when is", "why's": "why is",
    "how's": "how is", "it's": "it is", "he's": "he is",
    "she's": "she is", "that's": "that is", "there's": "there is",
    "they're": "they are", "we're": "we are", "you're": "you are",
    "i've": "i have", "you've": "you have", "we've": "we have",
    "they've": "they have", "i'd": "i would", "you'd": "you would",
    "he'd": "he would", "she'd": "she would", "we'd": "we would",
    "they'd": "they would", "i'll": "i will", "you'll": "you will",
    "he'll": "he will", "she'll": "she will", "we'll": "we will",
    "they'll": "they will", "i'm": "i am", "can't": "cannot",
    "couldn't": "could not", "don't": "do not", "doesn't": "does not",
    "didn't": "did not", "isn't": "is not", "aren't": "are not",
    "wasn't": "was not", "weren't": "were not", "won't": "will not",
    "wouldn't": "would not", "haven't": "have not", "hasn't": "has not",
    "hadn't": "had not", "should've": "should have",
    "would've": "would have", "could've": "could have",
}


def clean_for_lstm(text: str) -> str:
    """Lowercase, expand common contractions, keep English letters and spaces only."""
    if not isinstance(text, str) or text == "":
        return ""
    text = text.lower()
    for src, dst in CONTRACTIONS.items():
        text = text.replace(src, dst)
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def clean_for_bert(text: str) -> str:
    """Light cleaning backup for teammates who use transformer models."""
    if not isinstance(text, str) or text == "":
        return ""
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def encode_pad(texts: pd.Series, word2idx: dict[str, int], max_len: int) -> np.ndarray:
    """Encode text as right-padded integer sequences.

    Empty questions get one <UNK> token instead of becoming all padding.
    This avoids edge cases in GPU LSTM execution.
    """
    result = np.zeros((len(texts), max_len), dtype=np.int32)
    for row_idx, text in enumerate(texts):
        tokens = str(text).split()[:max_len]
        ids = [word2idx.get(token, UNK_ID) for token in tokens]
        if not ids:
            ids = [UNK_ID]
        result[row_idx, : len(ids)] = ids
    return result


def have_required_lstm_files() -> bool:
    return all((LSTM_DIR / filename).exists() for filename in REQUIRED_LSTM_FILES)


def save_split_table(name: str, split_df: pd.DataFrame) -> None:
    """Save raw and cleaned split data for later analysis and plotting."""
    columns = [
        "original_index", "id", "qid1", "qid2",
        "question1", "question2", "q1_lstm", "q2_lstm",
        "q1_bert", "q2_bert", "is_duplicate",
    ]
    existing_columns = [col for col in columns if col in split_df.columns]
    split_df[existing_columns].to_csv(TABLE_DIR / f"{name}_split_text.csv", index=False)


def build_lstm_preprocessing(data_path: Path) -> None:
    print("Building clean LSTM preprocessing outputs from:", data_path)
    df = pd.read_csv(data_path)
    print("Raw data shape:", df.shape)

    required_cols = {"question1", "question2", "is_duplicate"}
    missing_cols = required_cols - set(df.columns)
    if missing_cols:
        raise ValueError(f"Missing required columns in train.csv: {missing_cols}")

    df = df.copy()
    df["original_index"] = np.arange(len(df), dtype=np.int32)
    df["question1"] = df["question1"].fillna("")
    df["question2"] = df["question2"].fillna("")
    df["is_duplicate"] = df["is_duplicate"].astype(np.int32)

    print("Cleaning text...")
    df["q1_lstm"] = df["question1"].apply(clean_for_lstm)
    df["q2_lstm"] = df["question2"].apply(clean_for_lstm)
    df["q1_bert"] = df["question1"].apply(clean_for_bert)
    df["q2_bert"] = df["question2"].apply(clean_for_bert)

    idx = df.index.to_numpy()
    y = df["is_duplicate"].to_numpy()
    idx_train, idx_temp, _, y_temp = train_test_split(
        idx,
        y,
        test_size=0.2,
        random_state=RANDOM_STATE,
        stratify=y,
    )
    idx_val, idx_test, _, _ = train_test_split(
        idx_temp,
        y_temp,
        test_size=0.5,
        random_state=RANDOM_STATE,
        stratify=y_temp,
    )

    np.save(SPLIT_DIR / "train_idx.npy", idx_train)
    np.save(SPLIT_DIR / "val_idx.npy", idx_val)
    np.save(SPLIT_DIR / "test_idx.npy", idx_test)

    train_df = df.iloc[idx_train].reset_index(drop=True)
    val_df = df.iloc[idx_val].reset_index(drop=True)
    test_df = df.iloc[idx_test].reset_index(drop=True)

    print(f"Split sizes: train={len(train_df):,}, val={len(val_df):,}, test={len(test_df):,}")

    print("Building vocabulary from train split only...")
    word_freq: Counter[str] = Counter()
    for col in ["q1_lstm", "q2_lstm"]:
        for text in train_df[col].astype(str):
            word_freq.update(text.split())

    vocab = [PAD_TOKEN, UNK_TOKEN]
    vocab += [
        word for word, count in word_freq.most_common()
        if count >= MIN_WORD_COUNT and word not in {PAD_TOKEN, UNK_TOKEN}
    ]
    word2idx = {word: idx for idx, word in enumerate(vocab)}
    idx2word = {idx: word for word, idx in word2idx.items()}

    with open(LSTM_DIR / "word2idx.pkl", "wb") as f:
        pickle.dump(word2idx, f)
    with open(LSTM_DIR / "idx2word.pkl", "wb") as f:
        pickle.dump(idx2word, f)

    split_frames = {"train": train_df, "val": val_df, "test": test_df}
    for name, split_df in split_frames.items():
        q1 = encode_pad(split_df["q1_lstm"], word2idx, MAX_SEQ_LEN)
        q2 = encode_pad(split_df["q2_lstm"], word2idx, MAX_SEQ_LEN)
        labels = split_df["is_duplicate"].to_numpy(dtype=np.int32)

        np.save(LSTM_DIR / f"{name}_q1.npy", q1)
        np.save(LSTM_DIR / f"{name}_q2.npy", q2)
        np.save(LSTM_DIR / f"{name}_y.npy", labels)
        save_split_table(name, split_df)

    # Backup CSVs for transformer-based teammates.
    bert_cols = ["original_index", "question1", "question2", "q1_bert", "q2_bert", "is_duplicate"]
    for optional in ["id", "qid1", "qid2"]:
        if optional in df.columns and optional not in bert_cols:
            bert_cols.insert(1, optional)
    bert_cols = [col for col in bert_cols if col in df.columns]
    train_df[bert_cols].to_csv(BERT_DIR / "train.csv", index=False)
    val_df[bert_cols].to_csv(BERT_DIR / "val.csv", index=False)
    test_df[bert_cols].to_csv(BERT_DIR / "test.csv", index=False)

    class_dist_rows = []
    for name, split_df in split_frames.items():
        counts = split_df["is_duplicate"].value_counts().sort_index()
        total = len(split_df)
        for label in [0, 1]:
            class_dist_rows.append({
                "split": name,
                "label": label,
                "count": int(counts.get(label, 0)),
                "ratio": float(counts.get(label, 0) / total),
            })
    class_dist_df = pd.DataFrame(class_dist_rows)
    class_dist_df.to_csv(TABLE_DIR / "class_distribution.csv", index=False)

    preprocessing_summary = {
        "raw_rows": int(len(df)),
        "train_rows": int(len(train_df)),
        "val_rows": int(len(val_df)),
        "test_rows": int(len(test_df)),
        "vocabulary_size": int(len(word2idx)),
        "max_sequence_length": int(MAX_SEQ_LEN),
        "min_word_count": int(MIN_WORD_COUNT),
        "pad_token": PAD_TOKEN,
        "pad_id": PAD_ID,
        "unk_token": UNK_TOKEN,
        "unk_id": UNK_ID,
    }
    pd.DataFrame([preprocessing_summary]).to_csv(TABLE_DIR / "preprocessing_summary.csv", index=False)
    with open(META_DIR / "preprocessing_summary.json", "w", encoding="utf-8") as f:
        json.dump(preprocessing_summary, f, indent=2, ensure_ascii=False)

    print("Saved preprocessing arrays and tables.")


if REBUILD_PREPROCESS or not have_required_lstm_files():
    build_lstm_preprocessing(DATA_PATH)
else:
    print("Using existing outputs/for_lstm. Set REBUILD_PREPROCESS=True for a clean rebuild.")

In [ ]:
# =========================
# 4. Load arrays and split tables

In [ ]:
# =========================

train_q1 = np.load(LSTM_DIR / "train_q1.npy")
train_q2 = np.load(LSTM_DIR / "train_q2.npy")
y_train = np.load(LSTM_DIR / "train_y.npy")
val_q1 = np.load(LSTM_DIR / "val_q1.npy")
val_q2 = np.load(LSTM_DIR / "val_q2.npy")
y_val = np.load(LSTM_DIR / "val_y.npy")
test_q1 = np.load(LSTM_DIR / "test_q1.npy")
test_q2 = np.load(LSTM_DIR / "test_q2.npy")
y_test = np.load(LSTM_DIR / "test_y.npy")

with open(LSTM_DIR / "word2idx.pkl", "rb") as f:
    word2idx = pickle.load(f)
with open(LSTM_DIR / "idx2word.pkl", "rb") as f:
    idx2word = pickle.load(f)

array_max = int(max(train_q1.max(), train_q2.max(), val_q1.max(), val_q2.max(), test_q1.max(), test_q2.max()))
VOCAB_SIZE = max(max(word2idx.values()) + 1, array_max + 1)
MAX_LEN = train_q1.shape[1]
PAD_ID = int(word2idx.get(PAD_TOKEN, 0))
UNK_ID = int(word2idx.get(UNK_TOKEN, 1))

split_shape_rows = [
    {"split": "train", "q1_shape": str(train_q1.shape), "q2_shape": str(train_q2.shape), "label_shape": str(y_train.shape)},
    {"split": "val", "q1_shape": str(val_q1.shape), "q2_shape": str(val_q2.shape), "label_shape": str(y_val.shape)},
    {"split": "test", "q1_shape": str(test_q1.shape), "q2_shape": str(test_q2.shape), "label_shape": str(y_test.shape)},
]
pd.DataFrame(split_shape_rows).to_csv(TABLE_DIR / "array_shapes.csv", index=False)

print("Loaded arrays:")
print("  train_q1:", train_q1.shape, "train_q2:", train_q2.shape, "y_train:", y_train.shape)
print("  val_q1  :", val_q1.shape, "val_q2  :", val_q2.shape, "y_val  :", y_val.shape)
print("  test_q1 :", test_q1.shape, "test_q2 :", test_q2.shape, "y_test :", y_test.shape)
print("  vocab_size:", VOCAB_SIZE, "max_len:", MAX_LEN)

In [ ]:
# =========================
# 5. Imports for modeling

In [ ]:
# =========================


def pip_install_if_missing(import_name: str, package_name: str | None = None) -> bool:
    package_name = package_name or import_name
    try:
        __import__(import_name)
        return True
    except Exception as first_error:
        print(f"Installing {package_name} because importing {import_name} failed: {first_error}")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package_name], check=False)
        try:
            __import__(import_name)
            return True
        except Exception as second_error:
            print(f"Could not import {import_name}: {second_error}")
            return False


GENSIM_OK = pip_install_if_missing("gensim", "gensim") if USE_WORD2VEC else False

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    log_loss,
    precision_score,
    recall_score,
)
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow.keras import Model, layers
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

try:
    gpu_summary = subprocess.getoutput("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader")
except Exception as e:
    gpu_summary = f"nvidia-smi unavailable: {e}"

ENV_INFO = {
    "python_version": sys.version,
    "platform": platform.platform(),
    "tensorflow_version": tf.__version__,
    "gpu_summary": gpu_summary,
    "working_directory": str(ROOT),
}
with open(META_DIR / "environment.json", "w", encoding="utf-8") as f:
    json.dump(ENV_INFO, f, indent=2, ensure_ascii=False)
with open(META_DIR / "run_config.json", "w", encoding="utf-8") as f:
    json.dump(CONFIG, f, indent=2, ensure_ascii=False)

print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))
print("GPU summary:", gpu_summary)

In [ ]:
# =========================
# 6. Word2Vec embedding matrix

In [ ]:
# =========================


def seq_to_tokens(seq: np.ndarray) -> list[str]:
    tokens: list[str] = []
    for raw_idx in seq:
        idx = int(raw_idx)
        if idx == PAD_ID:
            continue
        token = idx2word.get(idx, UNK_TOKEN)
        if token in {PAD_TOKEN, UNK_TOKEN, ""}:
            continue
        tokens.append(token)
    return tokens


embedding_matrix = None
embedding_source = "trainable_random_embedding"
word2vec_summary: dict[str, object] = {
    "embedding_source": embedding_source,
    "word2vec_success": False,
}

if USE_WORD2VEC and GENSIM_OK:
    from gensim.models import Word2Vec

    pair_limit = len(train_q1) if W2V_PAIR_LIMIT is None else min(int(W2V_PAIR_LIMIT), len(train_q1))
    print(f"Training Word2Vec from {pair_limit:,} training pairs...")

    w2v_sentences = []
    for seq in train_q1[:pair_limit]:
        w2v_sentences.append(seq_to_tokens(seq))
    for seq in train_q2[:pair_limit]:
        w2v_sentences.append(seq_to_tokens(seq))

    start_w2v = time.time()
    try:
        w2v_model = Word2Vec(
            sentences=w2v_sentences,
            vector_size=EMBED_DIM,
            window=W2V_WINDOW,
            min_count=W2V_MIN_COUNT,
            workers=max(1, os.cpu_count() or 1),
            sg=1,
            epochs=W2V_EPOCHS,
            seed=RANDOM_STATE,
        )

        embedding_matrix = np.random.normal(0.0, 0.05, size=(VOCAB_SIZE, EMBED_DIM)).astype(np.float32)
        if PAD_ID < VOCAB_SIZE:
            embedding_matrix[PAD_ID] = np.zeros(EMBED_DIM, dtype=np.float32)

        matched_words = 0
        for word, idx in word2idx.items():
            if idx < VOCAB_SIZE and word in w2v_model.wv:
                embedding_matrix[idx] = w2v_model.wv[word]
                matched_words += 1

        embedding_source = "word2vec_skipgram"
        word2vec_time_sec = time.time() - start_w2v
        np.save(ARRAY_DIR / "embedding_matrix.npy", embedding_matrix)
        w2v_model.save(str(MODEL_DIR / "word2vec.model"))

        word2vec_summary = {
            "embedding_source": embedding_source,
            "word2vec_success": True,
            "word2vec_time_sec": float(word2vec_time_sec),
            "word2vec_training_pairs": int(pair_limit),
            "word2vec_sentences": int(len(w2v_sentences)),
            "embedding_dim": int(EMBED_DIM),
            "vocabulary_size": int(VOCAB_SIZE),
            "matched_words": int(matched_words),
            "matched_ratio": float(matched_words / max(len(word2idx), 1)),
        }
        print("Word2Vec success:", word2vec_summary)

    except Exception as e:
        print("Word2Vec failed. Falling back to trainable random embedding.")
        print("Reason:", repr(e))
        embedding_matrix = None
        embedding_source = "trainable_random_embedding_fallback"
        word2vec_summary = {
            "embedding_source": embedding_source,
            "word2vec_success": False,
            "error": repr(e),
        }
else:
    print("Skipping Word2Vec. Using trainable random embedding.")

with open(META_DIR / "word2vec_summary.json", "w", encoding="utf-8") as f:
    json.dump(word2vec_summary, f, indent=2, ensure_ascii=False)
pd.DataFrame([word2vec_summary]).to_csv(TABLE_DIR / "word2vec_summary.csv", index=False)

In [ ]:
# =========================
# 7. Build Siamese LSTM

In [ ]:
# =========================

tf.keras.backend.clear_session()
tf.random.set_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

# Colab GPU uses cuDNN for LSTM. To avoid mask-related cuDNN assertion errors,
# keep mask_zero=False. PAD embedding is still initialized as zeros.
USE_MASK_ZERO = False

sentence_input = layers.Input(shape=(MAX_LEN,), dtype="int32", name="sentence_input")

if embedding_matrix is not None:
    emb = layers.Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBED_DIM,
        weights=[embedding_matrix],
        trainable=True,
        mask_zero=USE_MASK_ZERO,
        name="word2vec_embedding",
    )(sentence_input)
else:
    emb = layers.Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBED_DIM,
        trainable=True,
        mask_zero=USE_MASK_ZERO,
        name="trainable_embedding",
    )(sentence_input)

x = layers.SpatialDropout1D(0.20, name="embedding_dropout")(emb)
sentence_vector = layers.LSTM(LSTM_UNITS, name="shared_lstm")(x)
encoder = Model(sentence_input, sentence_vector, name="shared_lstm_encoder")

q1_input = layers.Input(shape=(MAX_LEN,), dtype="int32", name="q1_input")
q2_input = layers.Input(shape=(MAX_LEN,), dtype="int32", name="q2_input")
q1_vec = encoder(q1_input)
q2_vec = encoder(q2_input)

abs_diff = layers.Lambda(lambda z: tf.abs(z[0] - z[1]), name="abs_diff")([q1_vec, q2_vec])
elementwise_mul = layers.Multiply(name="elementwise_mul")([q1_vec, q2_vec])
features = layers.Concatenate(name="pair_features")([q1_vec, q2_vec, abs_diff, elementwise_mul])

x = layers.Dense(DENSE_UNITS, activation="relu", name="dense_1")(features)
x = layers.Dropout(DROPOUT_RATE, name="dropout_1")(x)
x = layers.Dense(64, activation="relu", name="dense_2")(x)
x = layers.Dropout(0.20, name="dropout_2")(x)
output = layers.Dense(1, activation="sigmoid", name="duplicate_probability")(x)

model = Model(inputs=[q1_input, q2_input], outputs=output, name="word2vec_siamese_lstm")
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

model_architecture = {
    "model_name": model.name,
    "encoder_name": encoder.name,
    "embedding_source": embedding_source,
    "vocab_size": int(VOCAB_SIZE),
    "embedding_dim": int(EMBED_DIM),
    "max_seq_len": int(MAX_LEN),
    "lstm_units": int(LSTM_UNITS),
    "dense_units": int(DENSE_UNITS),
    "dropout_rate": float(DROPOUT_RATE),
    "mask_zero": bool(USE_MASK_ZERO),
    "parameter_count": int(model.count_params()),
}
with open(META_DIR / "model_architecture.json", "w", encoding="utf-8") as f:
    json.dump(model_architecture, f, indent=2, ensure_ascii=False)
print("Model parameter count:", model.count_params())

In [ ]:
# =========================
# 8. Train

In [ ]:
# =========================

if TRAIN_LIMIT is None:
    X1_train = train_q1
    X2_train = train_q2
    yy_train = y_train
else:
    n = min(int(TRAIN_LIMIT), len(train_q1))
    X1_train = train_q1[:n]
    X2_train = train_q2[:n]
    yy_train = y_train[:n]

classes = np.array([0, 1])
class_weights_values = compute_class_weight(class_weight="balanced", classes=classes, y=yy_train)
class_weight = {int(cls): float(weight) for cls, weight in zip(classes, class_weights_values)}

callbacks = [
    EarlyStopping(monitor="val_loss", patience=PATIENCE, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=1, min_lr=1e-5, verbose=1),
    ModelCheckpoint(filepath=str(MODEL_DIR / "best_siamese_lstm.keras"), monitor="val_loss", save_best_only=True, verbose=1),
]

print("Training samples:", len(yy_train))
print("Class weight:", class_weight)
start_train = time.time()
history = model.fit(
    [X1_train, X2_train],
    yy_train,
    validation_data=([val_q1, val_q2], y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    class_weight=class_weight,
    callbacks=callbacks,
    verbose=1,
)
training_time_sec = time.time() - start_train
print(f"Training time: {training_time_sec:.2f} sec")

history_df = pd.DataFrame(history.history)
history_df.insert(0, "epoch", np.arange(1, len(history_df) + 1))
history_df.to_csv(TABLE_DIR / "training_history.csv", index=False)

In [ ]:
# =========================
# 9. Predict, tune threshold, evaluate

In [ ]:
# =========================


def predict_probs(q1: np.ndarray, q2: np.ndarray) -> np.ndarray:
    return model.predict([q1, q2], batch_size=BATCH_SIZE, verbose=1).ravel()


print("Predicting validation probabilities...")
val_prob = predict_probs(val_q1, val_q2)
np.save(ARRAY_DIR / "val_probabilities.npy", val_prob.astype(np.float32))

threshold_rows = []
for threshold in np.arange(0.20, 0.81, 0.01):
    pred = (val_prob >= threshold).astype(np.int32)
    threshold_rows.append({
        "threshold": float(threshold),
        "val_accuracy": accuracy_score(y_val, pred),
        "val_f1": f1_score(y_val, pred),
        "val_precision": precision_score(y_val, pred, zero_division=0),
        "val_recall": recall_score(y_val, pred, zero_division=0),
    })
threshold_df = pd.DataFrame(threshold_rows)
threshold_df.to_csv(TABLE_DIR / "threshold_search_validation.csv", index=False)

best_f1_idx = threshold_df["val_f1"].idxmax()
best_acc_idx = threshold_df["val_accuracy"].idxmax()
best_f1_threshold = float(threshold_df.loc[best_f1_idx, "threshold"])
best_acc_threshold = float(threshold_df.loc[best_acc_idx, "threshold"])

print("Best validation F1 threshold:", best_f1_threshold)
print("Best validation accuracy threshold:", best_acc_threshold)

print("Predicting test probabilities...")
start_infer = time.time()
test_prob = predict_probs(test_q1, test_q2)
inference_time_sec = time.time() - start_infer
np.save(ARRAY_DIR / "test_probabilities.npy", test_prob.astype(np.float32))

# Candidate threshold comparison for teammates.
threshold_candidates = [
    ("val_f1_best", best_f1_threshold),
    ("default_0_50", 0.50),
    ("val_accuracy_best", best_acc_threshold),
]
threshold_comparison_rows = []
for threshold_name, threshold in threshold_candidates:
    val_pred = (val_prob >= threshold).astype(np.int32)
    test_pred = (test_prob >= threshold).astype(np.int32)
    threshold_comparison_rows.append({
        "threshold_type": threshold_name,
        "threshold": float(threshold),
        "val_accuracy": accuracy_score(y_val, val_pred),
        "val_f1": f1_score(y_val, val_pred),
        "val_precision": precision_score(y_val, val_pred, zero_division=0),
        "val_recall": recall_score(y_val, val_pred, zero_division=0),
        "test_accuracy": accuracy_score(y_test, test_pred),
        "test_f1": f1_score(y_test, test_pred),
        "test_precision": precision_score(y_test, test_pred, zero_division=0),
        "test_recall": recall_score(y_test, test_pred, zero_division=0),
    })
threshold_comparison_df = pd.DataFrame(threshold_comparison_rows)
threshold_comparison_df.to_csv(TABLE_DIR / "threshold_comparison_val_test.csv", index=False)

# Official Jun result uses the validation-F1 threshold.
test_pred_best_f1 = (test_prob >= best_f1_threshold).astype(np.int32)
test_pred_default = (test_prob >= 0.50).astype(np.int32)
test_pred_best_acc = (test_prob >= best_acc_threshold).astype(np.int32)

# Main metrics.
test_accuracy = accuracy_score(y_test, test_pred_best_f1)
test_f1 = f1_score(y_test, test_pred_best_f1)
test_precision = precision_score(y_test, test_pred_best_f1, zero_division=0)
test_recall = recall_score(y_test, test_pred_best_f1, zero_division=0)
test_log_loss = log_loss(y_test, test_prob)
inference_ms_per_sample = inference_time_sec / len(y_test) * 1000

# Confusion matrix and classification report.
cm = confusion_matrix(y_test, test_pred_best_f1)
cm_df = pd.DataFrame(
    cm,
    index=["true_not_duplicate", "true_duplicate"],
    columns=["pred_not_duplicate", "pred_duplicate"],
)
cm_df.to_csv(TABLE_DIR / "confusion_matrix_best_f1_threshold.csv")

cm_long_rows = []
for i, true_label in enumerate([0, 1]):
    for j, pred_label in enumerate([0, 1]):
        cm_long_rows.append({
            "true_label": true_label,
            "pred_label": pred_label,
            "count": int(cm[i, j]),
        })
pd.DataFrame(cm_long_rows).to_csv(TABLE_DIR / "confusion_matrix_long.csv", index=False)

report_dict = classification_report(y_test, test_pred_best_f1, output_dict=True, zero_division=0)
report_rows = []
for label, values in report_dict.items():
    if isinstance(values, dict):
        row = {"label": label}
        row.update(values)
        report_rows.append(row)
    else:
        report_rows.append({"label": label, "value": values})
classification_report_df = pd.DataFrame(report_rows)
classification_report_df.to_csv(TABLE_DIR / "classification_report_best_f1_threshold.csv", index=False)

In [ ]:
# =========================
# 10. Save reusable prediction tables

In [ ]:
# =========================


def error_type(y_true: int, y_pred: int) -> str:
    if y_true == 0 and y_pred == 0:
        return "TN"
    if y_true == 0 and y_pred == 1:
        return "FP"
    if y_true == 1 and y_pred == 0:
        return "FN"
    return "TP"


def attach_predictions(split_name: str, y_true: np.ndarray, prob: np.ndarray, include_text: bool) -> pd.DataFrame:
    base = pd.DataFrame({
        "row_in_split": np.arange(len(y_true), dtype=np.int32),
        "y_true": y_true.astype(np.int32),
        "prob_duplicate": prob.astype(float),
        "pred_label_val_f1_threshold": (prob >= best_f1_threshold).astype(np.int32),
        "pred_label_default_0_50": (prob >= 0.50).astype(np.int32),
        "pred_label_val_accuracy_threshold": (prob >= best_acc_threshold).astype(np.int32),
    })
    base["error_type_val_f1_threshold"] = [
        error_type(int(y), int(p))
        for y, p in zip(base["y_true"], base["pred_label_val_f1_threshold"])
    ]

    if include_text:
        split_text_path = TABLE_DIR / f"{split_name}_split_text.csv"
        if split_text_path.exists():
            text_df = pd.read_csv(split_text_path)
            keep_cols = [
                "original_index", "id", "qid1", "qid2",
                "question1", "question2", "q1_lstm", "q2_lstm",
            ]
            keep_cols = [col for col in keep_cols if col in text_df.columns]
            base = pd.concat([text_df[keep_cols].reset_index(drop=True), base], axis=1)
    return base


val_predictions_df = attach_predictions("val", y_val, val_prob, EXPORT_TEXT_PREDICTIONS)
test_predictions_df = attach_predictions("test", y_test, test_prob, EXPORT_TEXT_PREDICTIONS)
val_predictions_df.to_csv(TABLE_DIR / "val_predictions.csv", index=False)
test_predictions_df.to_csv(TABLE_DIR / "test_predictions.csv", index=False)

# Smaller error-only files for quick qualitative analysis.
test_predictions_df[test_predictions_df["error_type_val_f1_threshold"] == "FP"].to_csv(TABLE_DIR / "false_positive_examples.csv", index=False)
test_predictions_df[test_predictions_df["error_type_val_f1_threshold"] == "FN"].to_csv(TABLE_DIR / "false_negative_examples.csv", index=False)

In [ ]:
# =========================
# 11. Save model and benchmark metrics

In [ ]:
# =========================

final_model_path = MODEL_DIR / "siamese_lstm_final.keras"
encoder_model_path = MODEL_DIR / "shared_lstm_encoder.keras"
model.save(final_model_path)
encoder.save(encoder_model_path)

model_size_mb = final_model_path.stat().st_size / (1024 * 1024)
encoder_size_mb = encoder_model_path.stat().st_size / (1024 * 1024)
parameter_count = int(model.count_params())
encoder_parameter_count = int(encoder.count_params())

try:
    import psutil

    memory_usage_mb = psutil.Process(os.getpid()).memory_info().rss / (1024 * 1024)
except Exception:
    memory_usage_mb = None

metrics = {
    "Model": "Word2Vec + Siamese LSTM" if embedding_matrix is not None else "Trainable Embedding + Siamese LSTM",
    "Owner": "Jun Park",
    "Role": "Dense embedding + Siamese LSTM deep learning baseline",
    "Embedding Source": embedding_source,
    "Official Threshold Type": "validation_f1_best",
    "Best Threshold": best_f1_threshold,
    "Accuracy Threshold": best_acc_threshold,
    "Train Samples": int(len(yy_train)),
    "Validation Samples": int(len(y_val)),
    "Test Samples": int(len(y_test)),
    "Test Accuracy": float(test_accuracy),
    "Test F1": float(test_f1),
    "Test Precision": float(test_precision),
    "Test Recall": float(test_recall),
    "Test Log Loss": float(test_log_loss),
    "Training Time sec": float(training_time_sec),
    "Inference Time sec": float(inference_time_sec),
    "Inference ms/sample": float(inference_ms_per_sample),
    "Model Size MB": float(model_size_mb),
    "Encoder Size MB": float(encoder_size_mb),
    "Parameter Count": parameter_count,
    "Encoder Parameter Count": encoder_parameter_count,
    "Memory RSS MB": None if memory_usage_mb is None else float(memory_usage_mb),
    "Max Sequence Length": int(MAX_LEN),
    "Vocabulary Size": int(VOCAB_SIZE),
    "Embedding Dim": int(EMBED_DIM),
    "LSTM Units": int(LSTM_UNITS),
    "Batch Size": int(BATCH_SIZE),
    "Epochs Run": int(len(history_df)),
    "GPU": gpu_summary,
}
metrics_df = pd.DataFrame([metrics])
metrics_df.to_csv(TABLE_DIR / "metrics_summary.csv", index=False)
metrics_df.to_csv(RESULT_DIR / "siamese_lstm_results.csv", index=False)
with open(META_DIR / "metrics_summary.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2, ensure_ascii=False)

In [ ]:
# =========================
# 12. Pair-level embedding analysis data, no plot

In [ ]:
# =========================


def cosine_similarity_pairwise(a: np.ndarray, b: np.ndarray, eps: float = 1e-8) -> np.ndarray:
    numerator = np.sum(a * b, axis=1)
    denominator = np.linalg.norm(a, axis=1) * np.linalg.norm(b, axis=1) + eps
    return numerator / denominator


print("Encoding test question pairs for reusable embedding analysis...")
test_q1_vec = encoder.predict(test_q1, batch_size=BATCH_SIZE, verbose=1).astype(np.float32)
test_q2_vec = encoder.predict(test_q2, batch_size=BATCH_SIZE, verbose=1).astype(np.float32)

if EXPORT_FULL_TEST_EMBEDDINGS:
    np.save(ARRAY_DIR / "test_q1_lstm_embeddings.npy", test_q1_vec)
    np.save(ARRAY_DIR / "test_q2_lstm_embeddings.npy", test_q2_vec)

pair_cosine = cosine_similarity_pairwise(test_q1_vec, test_q2_vec)
pair_similarity_df = pd.DataFrame({
    "row_in_split": np.arange(len(y_test), dtype=np.int32),
    "y_true": y_test.astype(np.int32),
    "prob_duplicate": test_prob.astype(float),
    "pred_label_val_f1_threshold": test_pred_best_f1.astype(np.int32),
    "cosine_similarity_q1_q2": pair_cosine.astype(float),
    "cosine_distance_q1_q2": (1.0 - pair_cosine).astype(float),
})

# Attach original index only, not full text, to keep this table compact.
test_split_text_path = TABLE_DIR / "test_split_text.csv"
if test_split_text_path.exists():
    test_text_df = pd.read_csv(test_split_text_path, usecols=lambda col: col in {"original_index", "id"})
    pair_similarity_df = pd.concat([test_text_df.reset_index(drop=True), pair_similarity_df], axis=1)

pair_similarity_df.to_csv(TABLE_DIR / "pair_similarity_scores.csv", index=False)

summary_rows = []
for label, group in pair_similarity_df.groupby("y_true"):
    s = group["cosine_similarity_q1_q2"]
    summary_rows.append({
        "y_true": int(label),
        "count": int(len(group)),
        "mean": float(s.mean()),
        "std": float(s.std()),
        "median": float(s.median()),
        "q25": float(s.quantile(0.25)),
        "q75": float(s.quantile(0.75)),
        "min": float(s.min()),
        "max": float(s.max()),
    })
pd.DataFrame(summary_rows).to_csv(TABLE_DIR / "pair_similarity_summary.csv", index=False)

In [ ]:
# =========================
# 13. K-Means + PCA data, no plot

In [ ]:
# =========================

sample_n = min(int(KMEANS_SAMPLE_N), len(test_q1_vec))
print("Generating K-Means/PCA table from sample_n:", sample_n)
question_vectors_sample = test_q1_vec[:sample_n]

kmeans = KMeans(n_clusters=KMEANS_K, random_state=RANDOM_STATE, n_init=10)
cluster_labels = kmeans.fit_predict(question_vectors_sample)

pca = PCA(n_components=PCA_COMPONENTS, random_state=RANDOM_STATE)
pca_points = pca.fit_transform(question_vectors_sample)

np.save(ARRAY_DIR / "kmeans_question_vectors_sample.npy", question_vectors_sample.astype(np.float32))
np.save(ARRAY_DIR / "kmeans_cluster_labels.npy", cluster_labels.astype(np.int32))
np.save(ARRAY_DIR / "kmeans_cluster_centers.npy", kmeans.cluster_centers_.astype(np.float32))
np.save(ARRAY_DIR / "pca_points_kmeans_sample.npy", pca_points.astype(np.float32))
np.save(ARRAY_DIR / "pca_components.npy", pca.components_.astype(np.float32))

kmeans_pca_df = pd.DataFrame({
    "row_in_test_split": np.arange(sample_n, dtype=np.int32),
    "pca_1": pca_points[:, 0].astype(float),
    "pca_2": pca_points[:, 1].astype(float),
    "kmeans_cluster": cluster_labels.astype(np.int32),
    "y_true_pair_label": y_test[:sample_n].astype(np.int32),
    "prob_duplicate": test_prob[:sample_n].astype(float),
    "pred_label_val_f1_threshold": test_pred_best_f1[:sample_n].astype(np.int32),
})
if test_split_text_path.exists():
    test_idx_text_df = pd.read_csv(test_split_text_path, usecols=lambda col: col in {"original_index", "id"})
    kmeans_pca_df = pd.concat([test_idx_text_df.iloc[:sample_n].reset_index(drop=True), kmeans_pca_df], axis=1)
kmeans_pca_df.to_csv(TABLE_DIR / "kmeans_lstm_pca_points.csv", index=False)

cluster_summary_df = kmeans_pca_df.groupby("kmeans_cluster").agg(
    count=("kmeans_cluster", "size"),
    duplicate_rate=("y_true_pair_label", "mean"),
    mean_prob_duplicate=("prob_duplicate", "mean"),
    mean_pca_1=("pca_1", "mean"),
    mean_pca_2=("pca_2", "mean"),
).reset_index()
cluster_summary_df.to_csv(TABLE_DIR / "kmeans_cluster_summary.csv", index=False)

pca_summary = {
    "sample_n": int(sample_n),
    "kmeans_k": int(KMEANS_K),
    "pca_components": int(PCA_COMPONENTS),
    "pca_explained_variance_ratio": [float(x) for x in pca.explained_variance_ratio_],
    "pca_explained_variance_ratio_sum": float(np.sum(pca.explained_variance_ratio_)),
}
with open(META_DIR / "kmeans_pca_summary.json", "w", encoding="utf-8") as f:
    json.dump(pca_summary, f, indent=2, ensure_ascii=False)

In [ ]:
# =========================
# 14. Output README and final checklist

In [ ]:
# =========================

readme_text = f"""# Jun Park Siamese LSTM Outputs, Data-Only Version

This folder contains reusable data outputs for Jun Park's part: Word2Vec dense embedding + Siamese LSTM.
No plots are generated by this script. Use the CSV/NPY files below to make final figures later.

## Official benchmark file
- `outputs/results/siamese_lstm_results.csv`: one-row benchmark summary for the final team table.

## Main tables under `outputs/jun/tables/`
- `metrics_summary.csv`: final accuracy, F1, log loss, speed, size, parameters, GPU, config summary.
- `training_history.csv`: epoch-level train/validation loss and accuracy. Use this for loss/accuracy curves.
- `threshold_search_validation.csv`: validation metrics for thresholds 0.20 to 0.80.
- `threshold_comparison_val_test.csv`: comparison among validation-F1 threshold, default 0.50, and validation-accuracy threshold.
- `test_predictions.csv`: test-set probability, predicted labels, error type, and optionally original question text.
- `val_predictions.csv`: validation-set probability and predicted labels.
- `confusion_matrix_best_f1_threshold.csv`: confusion matrix for the official threshold.
- `classification_report_best_f1_threshold.csv`: precision/recall/F1 summary.
- `pair_similarity_scores.csv`: cosine similarity between q1/q2 LSTM embeddings per test pair.
- `pair_similarity_summary.csv`: duplicate vs non-duplicate similarity statistics.
- `kmeans_lstm_pca_points.csv`: PCA coordinates and K-Means labels for plotting embedding clusters.
- `kmeans_cluster_summary.csv`: cluster sizes and duplicate-rate/probability summaries.
- `class_distribution.csv`: train/val/test class distribution.
- `preprocessing_summary.csv`: split size, vocabulary size, sequence length, PAD/UNK info.

## Main arrays under `outputs/jun/arrays/`
- `val_probabilities.npy`, `test_probabilities.npy`: raw model probabilities.
- `embedding_matrix.npy`: Word2Vec-initialized embedding matrix if Word2Vec succeeded.
- `test_q1_lstm_embeddings.npy`, `test_q2_lstm_embeddings.npy`: full test-set LSTM embeddings if enabled.
- `kmeans_question_vectors_sample.npy`, `pca_points_kmeans_sample.npy`, `kmeans_cluster_labels.npy`: plot-ready cluster arrays.

## Models under `outputs/jun/models/`
- `siamese_lstm_final.keras`: final duplicate-question classifier.
- `shared_lstm_encoder.keras`: encoder that maps one question to a dense LSTM vector.
- `best_siamese_lstm.keras`: best checkpoint selected by validation loss.
- `word2vec.model`: trained Word2Vec model if Word2Vec succeeded.

Official threshold type: validation F1 best.
Official threshold: {best_f1_threshold:.2f}
"""
(META_DIR / "README_outputs.md").write_text(readme_text, encoding="utf-8")

final_files = [
    RESULT_DIR / "siamese_lstm_results.csv",
    TABLE_DIR / "metrics_summary.csv",
    TABLE_DIR / "training_history.csv",
    TABLE_DIR / "threshold_search_validation.csv",
    TABLE_DIR / "threshold_comparison_val_test.csv",
    TABLE_DIR / "test_predictions.csv",
    TABLE_DIR / "pair_similarity_scores.csv",
    TABLE_DIR / "pair_similarity_summary.csv",
    TABLE_DIR / "kmeans_lstm_pca_points.csv",
    TABLE_DIR / "kmeans_cluster_summary.csv",
    MODEL_DIR / "siamese_lstm_final.keras",
    MODEL_DIR / "shared_lstm_encoder.keras",
    META_DIR / "README_outputs.md",
]

checklist_rows = []
for file_path in final_files:
    checklist_rows.append({
        "relative_path": str(file_path.relative_to(ROOT)),
        "exists": file_path.exists(),
        "size_mb": float(file_path.stat().st_size / (1024 * 1024)) if file_path.exists() else 0.0,
    })
checklist_df = pd.DataFrame(checklist_rows)
checklist_df.to_csv(TABLE_DIR / "final_file_checklist.csv", index=False)

print("\nFinal key metrics:")
print(metrics_df.to_string(index=False))
print("\nSaved final-file checklist:", TABLE_DIR / "final_file_checklist.csv")
print("\nImportant outputs:")
for row in checklist_rows:
    print(f"  {'OK' if row['exists'] else 'MISSING'} | {row['relative_path']} | {row['size_mb']:.2f} MB")
print("\nDone. No plots were generated.")